In [3]:
import os

# What's in /kaggle/input?
print("=== Contents of /kaggle/input ===")
if os.path.isdir("/kaggle/input"):
    for item in os.listdir("/kaggle/input"):
        full_path = f"/kaggle/input/{item}"
        if os.path.isdir(full_path):
            print(f"  📁 {item}/")
            try:
                for subitem in os.listdir(full_path)[:10]:
                    print(f"     - {subitem}")
            except PermissionError:
                print(f"     (permission denied)")
        else:
            print(f"  📄 {item}")
else:
    print("  /kaggle/input does not exist")

print(f"\n=== Working directory ===")
print(f"  {os.getcwd()}")

print(f"\n=== /kaggle/working contents ===")
for item in os.listdir("/kaggle/working"):
    print(f"  {item}")

=== Contents of /kaggle/input ===
  📁 datasets/
     - arinkc

=== Working directory ===
  /kaggle/working

=== /kaggle/working contents ===
  .virtual_documents


In [4]:
import os

def show_tree(path, max_depth=4, current_depth=0):
    if current_depth >= max_depth:
        return
    try:
        items = sorted(os.listdir(path))
        for item in items:
            full = os.path.join(path, item)
            indent = "  " * current_depth
            if os.path.isdir(full):
                print(f"{indent}📁 {item}/")
                show_tree(full, max_depth, current_depth + 1)
            else:
                size = os.path.getsize(full)
                print(f"{indent}📄 {item} ({size:,} bytes)")
    except PermissionError:
        print(f"{'  ' * current_depth}(permission denied)")

show_tree("/kaggle/input")

📁 datasets/
  📁 arinkc/
    📁 llm-finetuning-project-code/
      📄 .gitignore (77 bytes)
      📄 README.md (3,578 bytes)
      📁 evaluation/
      📁 notebooks/
      📄 requirements.txt (0 bytes)
      📁 src/


In [5]:
import sys
sys.path.insert(0, "/kaggle/input/datasets/arinkc/llm-finetuning-project-code")

from src.data_filter import passes_filter
print("✅ Filter imported")

# Quick sanity check
test_row = {
    'func_code_string': '''def add_numbers(a, b):
    """Adds two numbers and returns the sum of these numbers."""
    result = a + b
    # some additional content to push code over 100 chars
    print(f"Result: {result}")
    return result''',
    'func_documentation_string': 'Adds two numbers and returns the sum of these numbers.'
}
result = passes_filter(test_row)
print(f"Test result: {result}")

✅ Filter imported
Test result: (True, 'ok')


In [6]:
!rm -rf /kaggle/working/repo
!git clone https://github.com/arinkc/llm-finetuning-project.git /kaggle/working/repo

import sys
sys.path.insert(0, "/kaggle/working/repo")

from src.data_filter import passes_filter
print("✅ Filter imported from cloned repo")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 48 (delta 14), reused 41 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 70.36 KiB | 1.85 MiB/s, done.
Resolving deltas: 100% (14/14), done.
✅ Filter imported from cloned repo


In [8]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ HF authenticated")

✅ HF authenticated


In [9]:
from datasets import load_dataset

print("Loading full CodeSearchNet Python subset (~450K examples)...")
print("First run: downloads ~1GB, takes 2-5 minutes.")

ds = load_dataset(
    "code-search-net/code_search_net",
    "python",
    split="train",
)

print(f"\n✅ Loaded {len(ds):,} examples")
print(f"Columns: {list(ds.features.keys())}")

Loading full CodeSearchNet Python subset (~450K examples)...
First run: downloads ~1GB, takes 2-5 minutes.


README.md: 0.00B [00:00, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]


✅ Loaded 412,178 examples
Columns: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url']


In [10]:
def filter_fn(row):
    passes, _ = passes_filter(row)
    return passes

print(f"Filtering {len(ds):,} examples...")
filtered_ds = ds.filter(filter_fn, num_proc=4, desc="Filtering")
print(f"\n✅ Passed: {len(filtered_ds):,} examples ({100*len(filtered_ds)/len(ds):.2f}%)")

Filtering 412,178 examples...


Filtering (num_proc=4):   0%|          | 0/412178 [00:00<?, ? examples/s]


✅ Passed: 126,357 examples (30.66%)


In [11]:
TARGET_SIZE = 25_000

if len(filtered_ds) > TARGET_SIZE:
    sampled_ds = filtered_ds.shuffle(seed=42).select(range(TARGET_SIZE))
else:
    sampled_ds = filtered_ds

print(f"Sampled dataset: {len(sampled_ds):,} examples")

Sampled dataset: 25,000 examples


In [17]:
import ast
import re

def strip_docstring(code: str) -> str:
    """Remove the top-level docstring from a Python function.
    
    Returns the code with the docstring removed. If parsing fails,
    falls back to a regex-based stripper.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return _strip_docstring_regex(code)
    
    if not tree.body or not isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef)):
        return code
    
    func = tree.body[0]
    if not func.body:
        return code
    
    first_stmt = func.body[0]
    is_docstring = (
        isinstance(first_stmt, ast.Expr)
        and isinstance(first_stmt.value, ast.Constant)
        and isinstance(first_stmt.value.value, str)
    )
    
    if not is_docstring:
        return code
    
    lines = code.split('\n')
    start_line = first_stmt.lineno - 1
    end_line = first_stmt.end_lineno - 1
    
    new_lines = lines[:start_line] + lines[end_line + 1:]
    
    # Check if function body is now empty; if so, add `pass`
    sig_line = lines[func.lineno - 1]
    indent = len(sig_line) - len(sig_line.lstrip()) + 4
    
    remaining_body = '\n'.join(new_lines[start_line:]).strip()
    if not remaining_body:
        new_lines.insert(start_line, ' ' * indent + 'pass')
    
    return '\n'.join(new_lines)


def _strip_docstring_regex(code: str) -> str:
    """Fallback regex stripper for code that doesn't parse cleanly."""
    pattern = r'(def\s+\w+\s*\([^)]*\)\s*(?:->\s*[^:]+)?:\s*\n\s*)(?:"""[^"]*?"""|\'\'\'[^\']*?\'\'\')'
    return re.sub(pattern, r'\1', code, count=1)


# Sanity check on a real example
example = sampled_ds[0]
print("=== ORIGINAL CODE (first 400 chars) ===")
print(example['func_code_string'][:400])
print("\n=== STRIPPED CODE (first 400 chars) ===")
print(strip_docstring(example['func_code_string'])[:400])
print("\n=== TARGET DOCSTRING ===")
print(example['func_documentation_string'])

=== ORIGINAL CODE (first 400 chars) ===
def run(self):
        has_npm = npm_installation_check()
        if has_npm:
            run_npm_install()
        else:
            print("Warning: npm not installed using prebuilded js files!",
                  file=sys.stderr)
        """
        Download npm packages required by package.json and extract required
        files from them
        """
        for js in JS_FILES:
            down

=== STRIPPED CODE (first 400 chars) ===
def run(self):
        has_npm = npm_installation_check()
        if has_npm:
            run_npm_install()
        else:
            print("Warning: npm not installed using prebuilded js files!",
                  file=sys.stderr)
        """
        Download npm packages required by package.json and extract required
        files from them
        """
        for js in JS_FILES:
            down

=== TARGET DOCSTRING ===
Download npm packages required by package.json and extract required
        files from them

In [18]:
ast_success = 0
ast_failure = 0
no_docstring = 0
errored = 0

for i in range(min(1000, len(sampled_ds))):
    code = sampled_ds[i]['func_code_string']
    try:
        tree = ast.parse(code)
        if (tree.body 
            and isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef))
            and tree.body[0].body
            and isinstance(tree.body[0].body[0], ast.Expr)
            and isinstance(tree.body[0].body[0].value, ast.Constant)
            and isinstance(tree.body[0].body[0].value.value, str)):
            ast_success += 1
        else:
            no_docstring += 1
    except SyntaxError:
        ast_failure += 1
    except Exception:
        errored += 1

print(f"AST parse + has docstring: {ast_success}")
print(f"AST parse but no docstring: {no_docstring}")
print(f"AST parse failed (using regex fallback): {ast_failure}")
print(f"Other error: {errored}")

<unknown>:21: SyntaxWarning: invalid escape sequence '\('
<unknown>:22: SyntaxWarning: invalid escape sequence '\/'
<unknown>:23: SyntaxWarning: invalid escape sequence '\('
<unknown>:19: SyntaxWarning: invalid escape sequence '\s'
<unknown>:3: SyntaxWarning: invalid escape sequence '\['


AST parse + has docstring: 991
AST parse but no docstring: 2
AST parse failed (using regex fallback): 7
Other error: 0


In [19]:
SYSTEM_PROMPT = """You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble."""


def format_example(row):
    code_without_doc = strip_docstring(row['func_code_string'])
    docstring = row['func_documentation_string'].strip()
    
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate a Google-style docstring for this function:\n\n```python\n{code_without_doc}\n```"},
            {"role": "assistant", "content": docstring},
        ]
    }


# Sanity check on one example
example = format_example(sampled_ds[0])
print("=== FORMATTED EXAMPLE ===")
for msg in example['messages']:
    print(f"\n[{msg['role'].upper()}]")
    content = msg['content']
    print(content[:400] + ("..." if len(content) > 400 else ""))

=== FORMATTED EXAMPLE ===

[SYSTEM]
You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble.

[USER]
Generate a Google-style docstring for this function:

```python
def run(self):
        has_npm = npm_installation_check()
        if has_npm:
            run_npm_install()
        else:
            print("Warning: npm not installed using prebuilded js files!",
                  file=sys.stderr)
        """
        Download npm packages required by package.json and extract required
        files fr...

[ASSISTANT]
Download npm packages required by package.json and extract required
        files from them


In [20]:
import ast
import re

def strip_docstring(code: str) -> tuple[str, bool]:
    """Remove the top-level docstring from a Python function.
    
    Returns:
        (stripped_code, success): The code with docstring removed, and a 
        bool indicating whether a docstring was actually found and removed.
    """
    try:
        tree = ast.parse(code)
    except SyntaxError:
        # Try the regex fallback
        stripped, found = _strip_docstring_regex(code)
        return stripped, found
    
    if not tree.body or not isinstance(tree.body[0], (ast.FunctionDef, ast.AsyncFunctionDef)):
        return code, False
    
    func = tree.body[0]
    if not func.body:
        return code, False
    
    first_stmt = func.body[0]
    is_docstring = (
        isinstance(first_stmt, ast.Expr)
        and isinstance(first_stmt.value, ast.Constant)
        and isinstance(first_stmt.value.value, str)
    )
    
    if not is_docstring:
        # Try harder: look for triple-quoted strings anywhere in the function body
        # that match the documentation_string we're targeting
        return code, False
    
    lines = code.split('\n')
    start_line = first_stmt.lineno - 1
    end_line = first_stmt.end_lineno - 1
    new_lines = lines[:start_line] + lines[end_line + 1:]
    
    # If we just emptied the function, add `pass`
    sig_line = lines[func.lineno - 1]
    indent = len(sig_line) - len(sig_line.lstrip()) + 4
    remaining = '\n'.join(new_lines[start_line:]).strip()
    if not remaining:
        new_lines.insert(start_line, ' ' * indent + 'pass')
    
    return '\n'.join(new_lines), True


def _strip_docstring_regex(code: str) -> tuple[str, bool]:
    """Regex fallback: removes the first triple-quoted string right after `def`."""
    pattern = r'(def\s+\w+\s*\([^)]*\)\s*(?:->\s*[^:]+)?:\s*\n\s*)(?:"""[\s\S]*?"""|\'\'\'[\s\S]*?\'\'\')'
    new_code, count = re.subn(pattern, r'\1', code, count=1)
    return new_code, count > 0

In [21]:
example = sampled_ds[0]
print("=== ORIGINAL ===")
print(example['func_code_string'])
print()

stripped, success = strip_docstring(example['func_code_string'])
print(f"=== STRIPPED (success={success}) ===")
print(stripped)

=== ORIGINAL ===
def run(self):
        has_npm = npm_installation_check()
        if has_npm:
            run_npm_install()
        else:
            print("Warning: npm not installed using prebuilded js files!",
                  file=sys.stderr)
        """
        Download npm packages required by package.json and extract required
        files from them
        """
        for js in JS_FILES:
            downloaded_js_name = os.path.join(TOP_DIR, js)
            installed_js_name = os.path.join(TOP_DIR, "sphinx_hwt", "html", js)
            if has_npm:
                assert os.path.exists(downloaded_js_name), downloaded_js_name
                os.makedirs(os.path.dirname(installed_js_name), exist_ok=True)
                copyfile(downloaded_js_name, installed_js_name)
                print("copy generated from NPM packages", installed_js_name)
            else:
                if os.path.exists(installed_js_name):
                    print("using prebuilded", installed_js_name)
 

In [22]:
success_count = 0
fail_count = 0
fail_examples = []

for i in range(min(1000, len(sampled_ds))):
    code = sampled_ds[i]['func_code_string']
    _, success = strip_docstring(code)
    if success:
        success_count += 1
    else:
        fail_count += 1
        if len(fail_examples) < 3:
            fail_examples.append((i, code))

print(f"Stripping success: {success_count}/{success_count + fail_count} ({100*success_count/(success_count+fail_count):.1f}%)")
print(f"Stripping failed:  {fail_count}")
print()

if fail_examples:
    print("=== EXAMPLES WHERE STRIPPING FAILED ===")
    for idx, code in fail_examples:
        print(f"--- Example {idx} ---")
        print(code[:400])
        print()

<unknown>:21: SyntaxWarning: invalid escape sequence '\('
<unknown>:22: SyntaxWarning: invalid escape sequence '\/'
<unknown>:23: SyntaxWarning: invalid escape sequence '\('
<unknown>:19: SyntaxWarning: invalid escape sequence '\s'
<unknown>:3: SyntaxWarning: invalid escape sequence '\['


Stripping success: 998/1000 (99.8%)
Stripping failed:  2

=== EXAMPLES WHERE STRIPPING FAILED ===
--- Example 0 ---
def run(self):
        has_npm = npm_installation_check()
        if has_npm:
            run_npm_install()
        else:
            print("Warning: npm not installed using prebuilded js files!",
                  file=sys.stderr)
        """
        Download npm packages required by package.json and extract required
        files from them
        """
        for js in JS_FILES:
            down

--- Example 11 ---
def wavelengthToRGB(wavelength):
	gamma = 0.80;
	intensityMax = 255;

	""" Taken from Earl F. Glynn's web page:
	* <a href="http://www.efg2.com/Lab/ScienceAndEngineering/Spectra.htm">Spectra Lab Report</a>
	"""

	factor = None
	r = None
	g = None
	b = None

	if((wavelength >= 380) and (wavelength<440)):
		r = -(wavelength - 440) / (440.0 - 380.0)
		g = 0.0
		b = 1.0
	elif((wavelength >= 440) and (



In [23]:
SYSTEM_PROMPT = """You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble."""


def format_example(row):
    code_without_doc, success = strip_docstring(row['func_code_string'])
    docstring = row['func_documentation_string'].strip()
    
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Generate a Google-style docstring for this function:\n\n```python\n{code_without_doc}\n```"},
            {"role": "assistant", "content": docstring},
        ],
        "_strip_success": success,
    }


# Apply
print(f"Formatting {len(sampled_ds):,} examples...")
formatted_ds = sampled_ds.map(
    format_example,
    remove_columns=sampled_ds.column_names,
    num_proc=4,
    desc="Formatting",
)

# Filter out failed strips
before = len(formatted_ds)
formatted_ds = formatted_ds.filter(lambda x: x['_strip_success'])
formatted_ds = formatted_ds.remove_columns(['_strip_success'])
after = len(formatted_ds)

print(f"✅ Formatted: {after:,} examples (dropped {before - after} failed strips)")

Formatting 25,000 examples...


Formatting (num_proc=4):   0%|          | 0/25000 [00:00<?, ? examples/s]

<unknown>:8: SyntaxWarning: invalid escape sequence '\e'
<unknown>:21: SyntaxWarning: invalid escape sequence '\('
<unknown>:22: SyntaxWarning: invalid escape sequence '\/'
<unknown>:23: SyntaxWarning: invalid escape sequence '\('
<unknown>:19: SyntaxWarning: invalid escape sequence '\s'
<unknown>:6: SyntaxWarning: invalid escape sequence '\w'
<unknown>:7: SyntaxWarning: invalid escape sequence '\s'
<unknown>:3: SyntaxWarning: invalid escape sequence '\s'
<unknown>:10: SyntaxWarning: invalid escape sequence '\*'
<unknown>:9: SyntaxWarning: invalid escape sequence '\s'
<unknown>:7: SyntaxWarning: invalid escape sequence '\s'
<unknown>:12: SyntaxWarning: invalid escape sequence '\('
<unknown>:3: SyntaxWarning: invalid escape sequence '\['
<unknown>:2: SyntaxWarning: invalid escape sequence '\s'
<unknown>:12: SyntaxWarning: invalid escape sequence '\A'
<unknown>:25: SyntaxWarning: invalid escape sequence '\%'
<unknown>:4: SyntaxWarning: invalid escape sequence '\w'
<unknown>:23: SyntaxWar

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

✅ Formatted: 24,970 examples (dropped 30 failed strips)


In [24]:
example = formatted_ds[0]
for msg in example['messages']:
    print(f"\n[{msg['role'].upper()}]")
    content = msg['content']
    print(content[:400] + ("..." if len(content) > 400 else ""))


[SYSTEM]
You are an expert Python documentation writer. Given a Python function, generate a concise, Google-style docstring. Output only the docstring text—no surrounding code, no markdown formatting, no preamble.

[USER]
Generate a Google-style docstring for this function:

```python
def address(self):
        self._address, value = self.get_attr_string(self._address, 'address')
        return value
```

[ASSISTANT]
Returns the name of the port that this motor is connected to.


In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.1-8B-Instruct",
    token=hf_token,
)
print("✅ Tokenizer loaded")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

✅ Tokenizer loaded


In [26]:
def compute_length(example):
    """Tokenize the full conversation and return total token count."""
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    tokens = tokenizer(text, return_tensors="pt")
    return {"num_tokens": tokens['input_ids'].shape[1]}


print("Tokenizing 2000-example sample for length analysis...")
sample_for_length = formatted_ds.shuffle(seed=42).select(range(min(2000, len(formatted_ds))))
sample_with_lengths = sample_for_length.map(compute_length, desc="Tokenizing")

import numpy as np
lengths = np.array(sample_with_lengths['num_tokens'])
print(f"\nToken length stats:")
print(f"  min:    {int(lengths.min())}")
print(f"  p25:    {int(np.percentile(lengths, 25))}")
print(f"  p50:    {int(np.percentile(lengths, 50))}")
print(f"  p75:    {int(np.percentile(lengths, 75))}")
print(f"  p90:    {int(np.percentile(lengths, 90))}")
print(f"  p95:    {int(np.percentile(lengths, 95))}")
print(f"  p99:    {int(np.percentile(lengths, 99))}")
print(f"  max:    {int(lengths.max())}")

Tokenizing 2000-example sample for length analysis...


Tokenizing:   0%|          | 0/2000 [00:00<?, ? examples/s]


Token length stats:
  min:    122
  p25:    165
  p50:    201
  p75:    267
  p90:    359
  p95:    421
  p99:    514
  max:    799


In [27]:
# Shuffle and split 90/5/5
split = formatted_ds.shuffle(seed=42).train_test_split(test_size=0.10, seed=42)
train_ds = split['train']

val_test_split = split['test'].train_test_split(test_size=0.5, seed=42)
val_ds = val_test_split['train']
test_ds = val_test_split['test']

print(f"Train:      {len(train_ds):,} examples")
print(f"Validation: {len(val_ds):,} examples")
print(f"Test:       {len(test_ds):,} examples")
print(f"Total:      {len(train_ds) + len(val_ds) + len(test_ds):,} examples")

Train:      22,473 examples
Validation: 1,248 examples
Test:       1,249 examples
Total:      24,970 examples


In [28]:
from datasets import DatasetDict

final_dataset = DatasetDict({
    'train': train_ds,
    'validation': val_ds,
    'test': test_ds,
})

HF_USERNAME = "Arinkc"
DATASET_NAME = "pydoc-llama-codesearchnet-curated"

repo_id = f"{HF_USERNAME}/{DATASET_NAME}"
print(f"Pushing to {repo_id}...")

final_dataset.push_to_hub(
    repo_id,
    private=False,
    commit_message="Curated Google-style Python docstring dataset from CodeSearchNet",
)
print(f"✅ Pushed: https://huggingface.co/datasets/{repo_id}")

Pushing to Arinkc/pydoc-llama-codesearchnet-curated...


Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Pushed: https://huggingface.co/datasets/Arinkc/pydoc-llama-codesearchnet-curated


In [29]:
final_dataset.save_to_disk("/kaggle/working/training_data")
print("✅ Saved to /kaggle/working/training_data")

# Also save a preview JSON for your repo
import json

preview = {
    "train_count": len(train_ds),
    "val_count": len(val_ds),
    "test_count": len(test_ds),
    "system_prompt": SYSTEM_PROMPT,
    "token_length_stats": {
        "p50": int(np.percentile(lengths, 50)),
        "p90": int(np.percentile(lengths, 90)),
        "p99": int(np.percentile(lengths, 99)),
        "max": int(lengths.max()),
    },
    "filter_stats": {
        "raw_codesearchnet": 412_178,
        "passed_filter": 126_357,
        "filter_pass_rate": 30.66,
        "sampled_for_training": 25_000,
        "dropped_failed_strip": 30,
        "final_dataset_size": 24_970,
    },
    "example_train": train_ds[0],
}

with open("/kaggle/working/dataset_preview.json", "w") as f:
    json.dump(preview, f, indent=2)
print("✅ Saved /kaggle/working/dataset_preview.json")

Saving the dataset (0/1 shards):   0%|          | 0/22473 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1248 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1249 [00:00<?, ? examples/s]

✅ Saved to /kaggle/working/training_data
✅ Saved /kaggle/working/dataset_preview.json
